## Time Series Spliting of the Data

In [ ]:
import sqlite3
import pandas as pd
from pathlib import Path

DB_PATH = 'chembl_36/chembl_36_sqlite/chembl_36.db' # or wherever your database is located

print(f"  Size: {Path(DB_PATH).stat().st_size / (1024**3):.1f} GB")

conn = sqlite3.connect(DB_PATH)

  Size: 27.7 GB


In [17]:
# Get all approved small molecule drugs with their canonical SMILES

query = """

SELECT DISTINCT
    md.molregno as drug_internal_id,
    md.chembl_id as drug_id,
    md.pref_name as drug_name,
    cs.canonical_smiles as smile
FROM molecule_dictionary md
JOIN compound_structures cs ON md.molregno = cs.molregno
WHERE 
  md.molecule_type = 'Small molecule'
  AND cs.canonical_smiles IS NOT NULL
  AND md.max_phase = 4
ORDER BY md.chembl_id;

"""

approved_drugs = pd.read_sql(query, conn)
print(f"Found {len(approved_drugs)} approved small molecule drugs")
print(approved_drugs.head().to_string(index=False))


Found 3127 approved small molecule drugs
 drug_internal_id      drug_id             drug_name                                          smile
           111185   CHEMBL1000            CETIRIZINE    O=C(O)COCCN1CCN(C(c2ccccc2)c2ccc(Cl)cc2)CC1
           165474 CHEMBL100116           PENTAZOCINE             CC(C)=CCN1CCC2(C)c3cc(O)ccc3CC1C2C
           111482   CHEMBL1002        LEVOSALBUTAMOL              CC(C)(C)NC[C@H](O)c1ccc(O)c(CO)c1
           111491   CHEMBL1003 CLAVULANATE POTASSIUM O=C([O-])[C@H]1/C(=C/CO)O[C@@H]2CC(=O)N21.[K+]
           111498   CHEMBL1004            DOXYLAMINE                 CN(C)CCOC(C)(c1ccccc1)c1ccccn1


In [18]:
second_query = """
SELECT
    md.molregno as drug_internal_id,
    md.chembl_id as drug_id,
    md.pref_name as drug_name,

    td.tid as protein_internal_id,
    td.chembl_id as protein_id,
    td.pref_name as protein_name,

    -- Binding strength
    MAX(act.pchembl_value) as pchembl_max,
    ROUND(AVG(act.pchembl_value), 2) as pchembl_avg,
    MIN(act.standard_value) as best_value,
    COUNT(DISTINCT act.activity_id) as num_measurements,

    -- Quality
    MAX(ass.confidence_score) as confidence,

    MIN(d.year) as first_published_year

FROM activities act
JOIN assays ass ON act.assay_id = ass.assay_id
JOIN target_dictionary td ON ass.tid = td.tid
JOIN molecule_dictionary md ON act.molregno = md.molregno
JOIN docs d ON act.doc_id = d.doc_id

WHERE td.target_type = 'SINGLE PROTEIN'
  -- AND td.organism = 'Homo sapiens'
  AND act.pchembl_value >= 5
  AND ass.confidence_score >= 7
  AND d.year IS NOT NULL  -- Ensure we have a valid date
  AND md.max_phase = 4

GROUP BY 
    md.molregno, md.chembl_id, md.pref_name,
    td.tid, td.chembl_id, td.pref_name

ORDER BY md.chembl_id, pchembl_max DESC;
"""

drugs_interactions = pd.read_sql(second_query, conn)
print(f"Found {len(drugs_interactions)} drug-protein interactions")
drugs_interactions



Found 15410 drug-protein interactions


,drug_internal_id,drug_id,drug_name,protein_internal_id,protein_id,protein_name,pchembl_max,pchembl_avg,best_value,num_measurements,confidence,first_published_year
0,111185,CHEMBL1000,CETIRIZINE,127,CHEMBL231,Histamine H1 receptor,8.23,7.50,5.89,12,9,2004
1,111185,CHEMBL1000,CETIRIZINE,102,CHEMBL1941,Histamine H2 receptor,5.73,5.73,1851.40,1,8,2023
2,111185,CHEMBL1000,CETIRIZINE,107,CHEMBL224,5-hydroxytryptamine receptor 2A,5.67,5.67,2117.80,1,8,2023
3,111185,CHEMBL1000,CETIRIZINE,227,CHEMBL1833,5-hydroxytryptamine receptor 2B,5.05,5.05,8927.30,1,8,2023
4,111185,CHEMBL1000,CETIRIZINE,155,CHEMBL238,Sodium-dependent dopamine transporter,5.04,5.04,9181.20,1,8,2023
...,...,...,...,...,...,...,...,...,...,...,...,...
15405,110803,CHEMBL998,LORATADINE,226,CHEMBL246,Beta-3 adrenergic receptor,5.07,5.07,8500.00,1,8,2023
15406,164035,CHEMBL99946,LEVOMILNACIPRAN,12198,CHEMBL313,Sodium-dependent serotonin transporter,8.07,8.07,8.50,2,9,1996
15407,164035,CHEMBL99946,LEVOMILNACIPRAN,100,CHEMBL222,Sodium-dependent noradrenaline transporter,7.98,7.59,10.50,3,9,2008
15408,164035,CHEMBL99946,LEVOMILNACIPRAN,121,CHEMBL228,Sodium-dependent serotonin transporter,6.50,6.50,320.00,2,9,2008


In [19]:
# 1. Configuration

df = drugs_interactions
SPLIT_YEAR = 2022  # Training: <= 2022, Testing: > 2022

# 2. Split the Data
# The 'first_published_year' comes from your new SQL query
train_df = df[df['first_published_year'] <= SPLIT_YEAR].copy()
raw_test_df = df[df['first_published_year'] > SPLIT_YEAR].copy()

# 3. Analyze "Cold Start" vs. "Repurposing"
# Get the set of Drugs and Proteins the model "knows" from training
known_drugs = set(train_df['drug_id'].unique())
known_proteins = set(train_df['protein_id'].unique())

# Separate valid and invalid edges
valid_test_df = raw_test_df[
    (raw_test_df['drug_id'].isin(known_drugs)) & 
    (raw_test_df['protein_id'].isin(known_proteins))
].copy()

invalid_test_df = raw_test_df[
    ~((raw_test_df['drug_id'].isin(known_drugs)) & 
      (raw_test_df['protein_id'].isin(known_proteins)))
].copy()

# 4. Add invalid edges to training set
drug_interactions_train = pd.concat([train_df, invalid_test_df], ignore_index=True)

# 5. Split valid test set into validation and test (50-50 split)
validation_size = len(valid_test_df) // 2
valid_test_df = valid_test_df.sample(frac=1, random_state=42).reset_index(drop=True)  # Shuffle
validation = valid_test_df.iloc[:validation_size].copy()
test = valid_test_df.iloc[validation_size:].copy()

# 6. Calculate Statistics
total_test_edges = len(raw_test_df)
valid_test_edges = len(valid_test_df)
invalid_test_edges = len(invalid_test_df)

print(f"--- 📅 Time-Split Analysis (Cutoff: {SPLIT_YEAR}) ---")
print(f"🔹 Original Training Edges (Historical): {len(train_df):,}")
print(f"➕ Invalid 'Cold Start' Edges added to Training: {invalid_test_edges:,}")
print(f"✅ Final Training Edges: {len(drug_interactions_train):,}")
print(f"\n🔹 Valid Test Edges (> {SPLIT_YEAR}): {valid_test_edges:,}")
print(f"📊 Validation Set: {len(validation):,}")
print(f"📊 Test Set: {len(test):,}")

# 7. Show Examples
if len(valid_test_df) > 0:
    top_repurposed_drug = valid_test_df['drug_name'].value_counts().idxmax()
    print(f"\nExample: Drug '{top_repurposed_drug}'")
    print(f"- Known edges in Training: {len(train_df[train_df['drug_name'] == top_repurposed_drug])}")
    print(f"- New edges found in Valid Test: {len(test[test['drug_name'] == top_repurposed_drug])}")    
    print(f"- New edges found in Validation: {len(validation[validation['drug_name'] == top_repurposed_drug])}")
   

--- 📅 Time-Split Analysis (Cutoff: 2022) ---
🔹 Original Training Edges (Historical): 10,409
➕ Invalid 'Cold Start' Edges added to Training: 1,199
✅ Final Training Edges: 11,608

🔹 Valid Test Edges (> 2022): 3,802
📊 Validation Set: 1,901
📊 Test Set: 1,901

Example: Drug 'SUNITINIB'
- Known edges in Training: 279
- New edges found in Valid Test: 22
- New edges found in Validation: 23


In [20]:
import pandas as pd

# 1. Check what kind of References exist (Why did your join fail?)
print("--- 🔍 Analyzing Indication References ---")
ref_types = pd.read_sql("""
    SELECT ref_type, COUNT(*) as count 
    FROM indication_refs 
    GROUP BY ref_type 
    ORDER BY count DESC
""", conn)
print(ref_types)
print("\n(If 'PubMed' is low/zero, that explains why joining to DOCS failed.)\n")

# 2. Compare Data Availability for the 3 Paths
print("--- 📊 Comparing Date Sources ---")

# Path A: Via Compound Records (Did it come from a paper?)
path_a_count = pd.read_sql("""
    SELECT COUNT(DISTINCT di.drugind_id) as count
    FROM drug_indication di
    JOIN compound_records cr ON di.record_id = cr.record_id
    JOIN docs d ON cr.doc_id = d.doc_id
    WHERE d.year IS NOT NULL
""", conn).iloc[0]['count']

# Path B: Via FDA Products (Orange Book)
path_b_count = pd.read_sql("""
    SELECT COUNT(DISTINCT md.molregno) as count
    FROM molecule_dictionary md
    JOIN formulations f ON md.molregno = f.molregno
    JOIN products p ON f.product_id = p.product_id
    WHERE p.approval_date IS NOT NULL
""", conn).iloc[0]['count']

# Path C: Via First Approval Column
path_c_count = pd.read_sql("""
    SELECT COUNT(DISTINCT molregno) as count
    FROM molecule_dictionary 
    WHERE first_approval IS NOT NULL
""", conn).iloc[0]['count']

print(f"Path A (Paper Year via Record): {path_a_count} records")
print(f"Path B (FDA Product Date):      {path_b_count} drugs")
print(f"Path C (First Approval Year):   {path_c_count} drugs")

--- 🔍 Analyzing Indication References ---
         ref_type  count
0  ClinicalTrials  52451
1        DailyMed  32892
2             ATC   3251
3            USAN   1875
4             EMA   1787
5             FDA    718
6             INN    657

(If 'PubMed' is low/zero, that explains why joining to DOCS failed.)

--- 📊 Comparing Date Sources ---
Path A (Paper Year via Record): 0 records
Path B (FDA Product Date):      2258 drugs
Path C (First Approval Year):   3789 drugs


In [21]:
# 3. Final Query to Get Drug Indications with Best Available Year of Approval

drug_indications = pd.read_sql("""
SELECT 
    md.molregno as drug_internal_id,
    md.chembl_id as drug_id,
    md.pref_name as drug_name,
    
    di.mesh_id as effect_id,
    di.mesh_heading as effect_name,
    
    di.max_phase_for_ind as indication_phase,
    
    -- 1. RENAMED COLUMN: The best available year
    COALESCE(
        CAST(STRFTIME('%Y', MIN(p.approval_date)) AS INTEGER), 
        md.first_approval,
        md.usan_year,
        2000
    ) as approval_year,

    -- 2. NEW COLUMN: Source of Information (Traceability)
    CASE 
        WHEN MIN(p.approval_date) IS NOT NULL THEN 'FDA Orange Book (Product)'
        WHEN md.first_approval IS NOT NULL THEN 'ChEMBL First Approval'
        WHEN md.usan_year IS NOT NULL THEN 'USAN Assignment'
        ELSE 'Default (2000)'
    END as source_of_information

FROM drug_indication di
JOIN molecule_dictionary md ON di.molregno = md.molregno

-- Path B: Join to Products to get FDA Date
LEFT JOIN formulations f ON md.molregno = f.molregno
LEFT JOIN products p ON f.product_id = p.product_id

WHERE di.mesh_id IS NOT NULL
  AND md.max_phase = 4  -- Strictly Approved Drugs
  AND di.max_phase_for_ind >= 4  -- Indication must also be approved

GROUP BY 
    md.molregno, md.chembl_id, md.pref_name,
    di.mesh_id, di.mesh_heading,
    di.max_phase_for_ind,
    md.first_approval,
    md.usan_year

ORDER BY md.chembl_id;
""", conn)

print(f"Found {len(drug_indications)} indications.")
print(drug_indications[['approval_year', 'source_of_information']].value_counts())
print(drug_indications.head())

Found 7086 indications.
approval_year  source_of_information    
1982           FDA Orange Book (Product)    2072
1996           FDA Orange Book (Product)     172
1991           FDA Orange Book (Product)     106
2018           FDA Orange Book (Product)     103
2013           FDA Orange Book (Product)     101
                                            ... 
1988           USAN Assignment                 1
1986           USAN Assignment                 1
1965           USAN Assignment                 1
2016           USAN Assignment                 1
1942           ChEMBL First Approval           1
Name: count, Length: 146, dtype: int64
   drug_internal_id       drug_id              drug_name effect_id  \
0            111185    CHEMBL1000             CETIRIZINE   D005132   
1            111185    CHEMBL1000             CETIRIZINE   D006967   
2            165474  CHEMBL100116            PENTAZOCINE   D010146   
3            111491    CHEMBL1003  CLAVULANATE POTASSIUM   D007239   
4      

In [22]:
search_terms = ["Pantoprazole"]  # add more variants if needed

pattern = "|".join(term.strip() for term in search_terms)
drug_indications[
    drug_indications["drug_name"].str.contains(pattern, case=False, na=False, regex=True)
].sort_values(["drug_name", "approval_year"])

,drug_internal_id,drug_id,drug_name,effect_id,effect_name,indication_phase,approval_year,source_of_information
4765,2197430,CHEMBL3989559,PANTOPRAZOLE SODIUM,D004941,Esophagitis,4,2000,FDA Orange Book (Product)
4766,2197430,CHEMBL3989559,PANTOPRAZOLE SODIUM,D005764,Gastroesophageal Reflux,4,2000,FDA Orange Book (Product)
4767,2197430,CHEMBL3989559,PANTOPRAZOLE SODIUM,D015043,Zollinger-Ellison Syndrome,4,2000,FDA Orange Book (Product)


In [23]:
# store the data

approved_drugs.to_pickle('approved_small_molecule_drugs_review.pkl', protocol=4)
print("✓ Saved to approved_small_molecule_drugs_review.pkl")

✓ Saved to approved_small_molecule_drugs_review.pkl


In [24]:
train_df.to_pickle('drug_protein_interactions_train_review.pkl', protocol=4)
validation.to_pickle('drug_protein_interactions_validation_review.pkl', protocol=4)
test.to_pickle('drug_protein_interactions_test_review.pkl', protocol=4)
print("✓ Saved train, validation, and test sets with review suffix")

✓ Saved train, validation, and test sets with review suffix


In [25]:
drug_indications.to_pickle('drug_indications_review.pkl', protocol=4)
print("✓ Saved drug indications with review suffix")

✓ Saved drug indications with review suffix


## Negative Samples for each drug

In [26]:
failed_indications = pd.read_sql("""
SELECT 
    md.chembl_id as drug_id,
    di.mesh_heading as effect_name,
    0 as label,
    di.max_phase_for_ind as negative_phase
FROM drug_indication di
JOIN molecule_dictionary md ON di.molregno = md.molregno
WHERE md.max_phase = 4        -- The drug is approved overall
  AND di.max_phase_for_ind < 4 -- But it failed to get approved for THIS specific disease
  AND di.mesh_heading IS NOT NULL;
""", conn)

print(f"Found {len(failed_indications)} Failed Clinical Trials to use as Hard Negatives!")

Found 31475 Failed Clinical Trials to use as Hard Negatives!


In [27]:

negatives_per_drug = failed_indications.groupby('drug_id').size().reset_index(name='num_negatives')

# 3. Print the statistics
print(f"Total approved drugs with verified negatives: {len(negatives_per_drug)}")
print("\n--- Statistics of Negatives per Drug ---")
print(negatives_per_drug['num_negatives'].describe())

Total approved drugs with verified negatives: 2480

--- Statistics of Negatives per Drug ---
count    2480.000000
mean       12.691532
std        23.809593
min         1.000000
25%         2.000000
50%         5.000000
75%        12.000000
max       437.000000
Name: num_negatives, dtype: float64


In [28]:
failed_indications['negative_phase'].value_counts()

negative_phase
 2.0    13185
 3.0    11433
 1.0     5989
 0.5      861
-1.0        7
Name: count, dtype: int64

In [29]:
hard_negatives = failed_indications[failed_indications['negative_phase'] == 3]
medium_negatives = failed_indications[failed_indications['negative_phase'] < 3]
print(f"Hard Negatives (Failed Phase 3): {len(hard_negatives)}")
print(f"Medium Negatives (Failed Phase 2 or less): {len(medium_negatives)}")

Hard Negatives (Failed Phase 3): 11433
Medium Negatives (Failed Phase 2 or less): 20042


In [30]:
hard_negatives['drug_id'].nunique()

1769

In [31]:
medium_negatives['drug_id'].nunique()

2210

In [32]:
# save verified faild indications to disk
failed_indications_hard = hard_negatives    
failed_indications_medium = medium_negatives
failed_indications.to_pickle('failed_indications_review.pkl', protocol=4)
failed_indications_hard.to_pickle('failed_indications_hard.pkl', protocol=4)
failed_indications_medium.to_pickle('failed_indications_medium.pkl', protocol=4)
print("✓ Saved failed indications to failed_indications_review.pkl")
print("✓ Saved hard failed indications to failed_indications_hard.pkl")
print("✓ Saved medium failed indications to failed_indications_medium.pkl")

✓ Saved failed indications to failed_indications_review.pkl
✓ Saved hard failed indications to failed_indications_hard.pkl
✓ Saved medium failed indications to failed_indications_medium.pkl


In [33]:
verified_negatives_time_aware = pd.read_sql("""
WITH inactives AS (
    SELECT 
        md.chembl_id as drug_id,
        td.chembl_id as protein_id,
        MIN(d.year) as published_year  -- Grab the earliest year this failed
    FROM activities act
    JOIN assays ass ON act.assay_id = ass.assay_id
    JOIN target_dictionary td ON ass.tid = td.tid
    JOIN molecule_dictionary md ON act.molregno = md.molregno
    JOIN docs d ON act.doc_id = d.doc_id -- Join docs for the year
    WHERE td.target_type = 'SINGLE PROTEIN'
      AND md.max_phase = 4
      AND (
          act.pchembl_value < 4.5 
          OR act.standard_value >= 10000 
          OR act.standard_relation IN ('>', '>=') 
          OR LOWER(act.standard_text_value) IN ('inactive', 'not active', 'no effect')
          OR LOWER(act.activity_comment) LIKE '%inactive%'
      )
    GROUP BY md.chembl_id, td.chembl_id
),
actives AS (
    SELECT DISTINCT md.chembl_id as drug_id, td.chembl_id as protein_id
    FROM activities act
    JOIN assays ass ON act.assay_id = ass.assay_id
    JOIN target_dictionary td ON ass.tid = td.tid
    JOIN molecule_dictionary md ON act.molregno = md.molregno
    WHERE td.target_type = 'SINGLE PROTEIN'
      AND md.max_phase = 4 
      AND act.pchembl_value >= 5.0
)
SELECT 
    i.drug_id, 
    i.protein_id,
    0 as label,
    i.published_year
FROM inactives i
LEFT JOIN actives a ON i.drug_id = a.drug_id AND i.protein_id = a.protein_id
WHERE a.drug_id IS NULL;
""", conn)

print(f"Found {len(verified_negatives_time_aware)} time-stamped negative edges!")

Found 143522 time-stamped negative edges!


In [34]:
negative_ids_time_aware = len(set(verified_negatives_time_aware['drug_id'].unique()))
negative_ids_time_aware

2219

In [35]:
verified_negatives_time_aware_train = verified_negatives_time_aware[
    (verified_negatives_time_aware['published_year'] <= 2022) |
    (verified_negatives_time_aware['published_year'].isna())
]
verified_negatives_time_aware_test = verified_negatives_time_aware[
    (verified_negatives_time_aware['published_year'] > 2022)
]
print(f"Training set size: {len(verified_negatives_time_aware_train)}")
print(f"Test set size: {len(verified_negatives_time_aware_test)}")

Training set size: 78486
Test set size: 65036


In [36]:
verified_negatives_time_aware_train.to_pickle('verified_negatives_time_aware_train.pkl', protocol=4)
verified_negatives_time_aware_test.to_pickle('verified_negatives_time_aware_test.pkl', protocol=4)
print("✓ Saved time-aware verified negatives for training and testing")

✓ Saved time-aware verified negatives for training and testing
